# 🔄 使用 Azure OpenAI（Responses API）构建基础代理工作流（.NET）

## 📋 工作流编排教程

本笔记本演示如何使用 Microsoft Agent Framework for .NET 和 Azure OpenAI（Responses API）构建复杂的<strong>代理工作流</strong>。你将学习创建多步业务流程，AI 代理通过结构化的编排模式协作完成复杂任务。

## 🎯 学习目标

### 🏗️ <strong>工作流架构基础</strong>
- <strong>工作流构建器</strong>：设计和编排复杂多步骤的 AI 流程
- <strong>代理协调</strong>：在工作流中协调多个专业代理
- **Azure OpenAI（Responses API）**：在工作流中利用 Azure OpenAI Responses API
- <strong>可视化工作流设计</strong>：创建并可视化工作流结构以增强理解

### 🔄 <strong>流程编排模式</strong>
- <strong>顺序处理</strong>：按逻辑顺序链接多个代理任务
- <strong>状态管理</strong>：维护工作流各阶段的上下文和数据流
- <strong>错误处理</strong>：实现稳健的错误恢复和工作流弹性
- <strong>性能优化</strong>：设计适用于企业级操作的高效工作流

### 🏢 <strong>企业工作流应用</strong>
- <strong>业务流程自动化</strong>：自动化复杂的组织工作流
- <strong>内容生产流水线</strong>：包含审核和批准阶段的编辑工作流
- <strong>客户服务自动化</strong>：多步骤客户咨询解决流程
- <strong>数据处理工作流</strong>：带有 AI 驱动转化的 ETL 工作流

## ⚙️ 前提条件与设置

### 📦 **必需的 NuGet 包**

本工作流演示使用若干关键的 .NET 包：

```xml
<!-- Core AI Framework -->
<PackageReference Include="Microsoft.Extensions.AI" Version="9.9.0" />

<!-- Azure OpenAI (Responses API) -->
<PackageReference Include="Azure.AI.OpenAI" Version="2.1.0" />
<PackageReference Include="Azure.Identity" Version="1.13.1" />

<!-- Agent Framework (Local Development) -->
<!-- Microsoft.Agents.AI.dll - Core agent abstractions -->
<!-- Microsoft.Agents.AI.OpenAI.dll - Azure OpenAI (Responses API) integration -->

<!-- Configuration and Environment -->
<PackageReference Include="DotNetEnv" Version="3.1.1" />
```

### 🔑 **Azure OpenAI 配置**

**环境设置（.env 文件）：**
```env
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com
AZURE_OPENAI_DEPLOYMENT=gpt-5-mini
```

**Azure OpenAI 访问：**
1. 在 Azure 门户创建 Azure OpenAI 资源
2. 部署一个模型（例如 `gpt-5-mini`）并记下部署名称
3. 使用 `az login` 登录并按上方示例配置环境变量

### 🏗️ <strong>工作流架构概览</strong>

```mermaid
graph TD
    A[工作流构建器] --> B[代理注册表]
    B --> C[工作流执行引擎]
    C --> D[Agent 1: 内容生成器]
    C --> E[Agent 2: 内容审查员] 
    D --> F[工作流结果]
    E --> F
    G[Azure OpenAI（响应 API）] --> D
    G --> E
```

**关键组件：**
- **WorkflowBuilder**：用于设计工作流的主要编排引擎
- **AIAgent**：具备特定能力的独立专业代理
- **Azure OpenAI 客户端**：Azure OpenAI Responses API 集成
- <strong>执行上下文</strong>：管理工作流阶段之间的状态和数据流

## 🎨 <strong>企业工作流设计模式</strong>

### 📝 <strong>内容生产工作流</strong>
```
User Request → Content Generation → Quality Review → Final Output
```

### 🔍 <strong>文档处理流水线</strong>
```
Document Input → Analysis → Extraction → Validation → Structured Output
```

### 💼 <strong>商业智能工作流</strong>
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

### 🤝 <strong>客户服务自动化</strong>
```
Customer Inquiry → Classification → Processing → Response Generation → Follow-up
```

## 🏢 <strong>企业收益</strong>

### 🎯 <strong>可靠性与可扩展性</strong>
- <strong>确定性执行</strong>：一致且可复现的工作流结果
- <strong>错误恢复</strong>：对任一工作流阶段故障的优雅处理
- <strong>性能监控</strong>：跟踪执行指标与优化机会
- <strong>资源管理</strong>：高效分配和利用 AI 模型资源

### 🔒 <strong>安全与合规</strong>
- <strong>安全认证</strong>：通过 `az login` 使用 Microsoft Entra ID 认证（AzureCliCredential）
- <strong>审计追踪</strong>：完整记录工作流执行与决策点日志
- <strong>访问控制</strong>：细粒度权限管理工作流执行与监控
- <strong>数据隐私</strong>：工作流中敏感信息的安全处理

### 📊 <strong>可观测性与管理</strong>
- <strong>可视化工作流设计</strong>：清晰表现流程流向与依赖关系
- <strong>执行监控</strong>：实时跟踪工作流进度与性能
- <strong>错误报告</strong>：详尽的错误分析与调试功能
- <strong>性能分析</strong>：用于优化和容量规划的指标

让我们一起构建你的首个企业级 AI 工作流吧！🚀


In [1]:
#r "nuget: Microsoft.Extensions.AI, 9.9.1"

Installed Packages Microsoft.Extensions.AI, 9.9.1

In [ ]:
#r "nuget: Azure.AI.OpenAI, 2.1.0"

Installed Packages System.ClientModel, 1.6.1

In [3]:
#r "nuget: Azure.Identity, 1.15.0"
#r "nuget: System.Linq.Async, 6.0.3"
#r "nuget: OpenTelemetry.Api, 1.0.0"
#r "nuget: OpenTelemetry.Api, 1.0.0"

Installed Packages Azure.Identity, 1.15.0 OpenTelemetry.Api, 1.0.1 System.Linq.Async, 6.0.3

In [5]:

#r "nuget: Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3

In [ ]:

#r "nuget: Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.2

In [7]:
#r "nuget: DotNetEnv, 3.1.1"

Installed Packages DotNetEnv, 3.1.1

In [8]:
// #r "nuget: Microsoft.Extensions.AI.OpenAI, 9.9.0-preview.1.25458.4"

In [ ]:
using System;
using System.ComponentModel;
using Azure.AI.OpenAI;
using Azure.Identity;
using Microsoft.Extensions.AI;
using Microsoft.Agents.AI;
using Microsoft.Agents.AI.Workflows;

In [10]:
 using DotNetEnv;

In [11]:
Env.Load("../../../.env");

In [ ]:
// Azure OpenAI with the Responses API (stable v1 endpoint). Sign in with `az login`.
var azureEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT") ?? throw new InvalidOperationException("AZURE_OPENAI_ENDPOINT is not set.");
var deployment = Environment.GetEnvironmentVariable("AZURE_OPENAI_DEPLOYMENT") ?? "gpt-5-mini";

In [ ]:
// The Azure OpenAI client is created directly from the endpoint and Azure CLI credential — no custom client options are required.

In [ ]:
var azureClient = new AzureOpenAIClient(new Uri(azureEndpoint), new AzureCliCredential());

In [15]:
const string ReviewerAgentName = "Concierge";
const string ReviewerAgentInstructions = @"
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. ";

In [16]:
const string FrontDeskAgentName = "FrontDesk";
const string FrontDeskAgentInstructions = @"""
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """;

In [ ]:
AIAgent reviewerAgent = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:ReviewerAgentName,instructions:ReviewerAgentInstructions);
AIAgent frontDeskAgent  = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:FrontDeskAgentName,instructions:FrontDeskAgentInstructions);

In [18]:
var workflow = new WorkflowBuilder(frontDeskAgent)
            .AddEdge(frontDeskAgent, reviewerAgent)
            .Build();

In [19]:
ChatMessage userMessage = new ChatMessage(ChatRole.User, [
	new TextContent("I would like to go to Paris.") 
]);

In [20]:
StreamingRun run = await InProcessExecution.StreamAsync(workflow, userMessage);

In [21]:
await run.TrySendMessageAsync(new TurnToken(emitEvents: true));
string id="";
string messageData="";
await foreach (WorkflowEvent evt in run.WatchStreamAsync().ConfigureAwait(false))
{
    if (evt is AgentRunUpdateEvent executorComplete)
    {
        if(id=="")
        {
            id=executorComplete.ExecutorId;
        }
        if(id==executorComplete.ExecutorId)
        {
            messageData+=executorComplete.Data.ToString();
        }
        else
        {
            id=executorComplete.ExecutorId;
        }
        // Console.WriteLine($"{executorComplete.ExecutorId}: {executorComplete.Data}");
    }
}

Console.WriteLine(messageData);

Visit the Louvre Museum. It's a must-see for art enthusiasts and history lovers.That recommendation is quite popular and likely to attract many tourists. To refine it for a more local and authentic experience, consider suggesting an alternative that focuses on smaller, lesser-known art venues or galleries. Look for places where local artists exhibit or community spaces that host cultural events. This approach allows travelers to connect with the local art scene more intimately, away from the typical tourist routes.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
